# Predicción de precios de casas en Quito y los valles

**Proyecto final – Aprendizaje Supervisado (USFQ, 202610)**

**Integrantes:** _Nombre 1, Nombre 2, Nombre 3_

**Repositorio:** https://github.com/marcejose-ec/usfq-supervisado-proyecto-final

---

**Contenido**
1. Introducción
2. Configuración
3. Carga de datos
4. Análisis exploratorio de datos (EDA)
5. Preparación para el modelado
6. Partición entrenamiento / prueba
7. Pipeline: preprocesamiento, selección de características y modelo
8. Optimización de hiperparámetros (Optuna + repeated k-fold CV)
9. Curvas de validación y de aprendizaje
10. Comparación estadística de dos técnicas
11. Evaluación final en el conjunto de prueba
12. Visualización del espacio de características con t-SNE
13. Conclusiones
14. Referencias

## 1. Introducción

La vivienda es, para la mayoría de los hogares ecuatorianos, la inversión más grande de su vida. Sin embargo, en Quito
no existe un índice público y actualizado de precios de vivienda a nivel de barrio: compradores, vendedores y
agentes inmobiliarios fijan precios principalmente por comparación informal con anuncios cercanos. Esto genera
asimetrías de información, sobreprecios y dificulta la toma de decisiones tanto de familias como de entidades
financieras que otorgan créditos hipotecarios.

El Distrito Metropolitano de Quito presenta además una particularidad geográfica: junto a la ciudad consolidada
(norte, centro y sur) han crecido los **valles** (Cumbayá, Tumbaco, Valle de los Chillos, entre otros), con
dinámicas de precio muy distintas: lotes más grandes, predominio de casas sobre departamentos y una fuerte
valorización en la última década. Un modelo que capture estas diferencias es útil para:

- **Compradores y vendedores:** obtener un precio de referencia objetivo antes de negociar.
- **Bancos y aseguradoras:** apoyar avalúos y detectar anuncios atípicos.
- **Planificación urbana:** entender qué características (ubicación, área, amenidades) explican el valor del suelo.

**Objetivo:** construir un modelo de regresión supervisada que prediga el precio de venta (USD) de una vivienda en
Quito y sus valles a partir de sus características (área, habitaciones, baños, parqueaderos, ubicación, etc.),
siguiendo una metodología rigurosa: pipelines de `scikit-learn`, selección de características tipo filtro,
optimización de hiperparámetros con *repeated k-fold cross-validation* y una comparación estadística formal entre
dos técnicas de aprendizaje automático.

_TODO: añadir 1–2 cifras de contexto con su referencia (p. ej. número de anuncios, precio promedio por m²)._

## 2. Configuración

`MODO_RAPIDO = True` reduce repeticiones y número de *trials* para probar el notebook de principio a fin en pocos
minutos. **Para los resultados finales usar `MODO_RAPIDO = False`.**

In [ ]:
# Uso de herramientas de IA: la estructura base de este notebook se generó con ayuda de Claude (Anthropic).
# Prompts utilizados (según lo requerido en la rúbrica):
#   - "TODO: copiar aquí los prompts usados"

# En Colab, Optuna no viene instalado por defecto
try:
    import optuna
except ImportError:
    %pip install -q optuna
    import optuna

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import SelectPercentile, VarianceThreshold, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.manifold import TSNE
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             r2_score, root_mean_squared_error)
from sklearn.model_selection import (RepeatedKFold, cross_val_score, learning_curve,
                                     train_test_split, validation_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # MLP con early stopping
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

# ---------------- Parámetros globales ----------------
SEED = 42
MODO_RAPIDO = False

# Ruta local o URL "raw" de GitHub del dataset
DATA_PATH = "data/casas_quito.csv"
DATA_URL = "https://raw.githubusercontent.com/marcejose-ec/usfq-supervisado-proyecto-final/main/data/casas_quito.csv"

RESULTADOS = Path("resultados")        # tablas de resultados (CSV/JSON) que se suben al repositorio
RESULTADOS.mkdir(exist_ok=True)

TARGET = "precio"
SCORING = "neg_root_mean_squared_error"   # sobre y = log(precio) → RMSLE (sklearn lo maximiza en negativo)

if MODO_RAPIDO:
    CV_OPT = RepeatedKFold(n_splits=3, n_repeats=1, random_state=SEED)      # dentro de Optuna
    CV_COMP = RepeatedKFold(n_splits=5, n_repeats=2, random_state=SEED)     # comparación estadística
    N_TRIALS = 10
else:
    CV_OPT = RepeatedKFold(n_splits=5, n_repeats=2, random_state=SEED)
    CV_COMP = RepeatedKFold(n_splits=10, n_repeats=10, random_state=SEED)
    N_TRIALS = 60

# Núcleos para paralelizar (-1 = todos). En un servidor compartido, fijar con la variable de entorno N_JOBS.
N_JOBS = int(os.environ.get("N_JOBS", -1))
rng = np.random.default_rng(SEED)
print("optuna", optuna.__version__, "| MODO_RAPIDO =", MODO_RAPIDO)

## 3. Carga de datos

7.151 anuncios de venta de casas y departamentos recolectados el 11–12 de noviembre de 2026 de icasas.ec, remax.com.ec, arriendayvende.com, trivo.com.ec y sitios de constructoras [10]. La fuente, el diccionario de variables y los pasos de limpieza previos están en `data/README.md`.

In [ ]:
ruta = DATA_PATH if Path(DATA_PATH).exists() else DATA_URL
df_raw = pd.read_csv(ruta)

# Si el dataset trae año de construcción en vez de antigüedad, se convierte
print(f"Filas: {df_raw.shape[0]:,} | Columnas: {df_raw.shape[1]}")
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all").T

## 4. Análisis exploratorio de datos (EDA)

El EDA se hace **antes y fuera del pipeline de modelado**: es descriptivo y sirve para tomar decisiones informadas
sobre limpieza, transformación del objetivo, codificación de variables y filtrado de fuentes. Las transformaciones que
dependen de los datos (imputación, escalado, codificación, selección) se ajustan después **dentro** del pipeline,
solo con datos de entrenamiento.

### 4.1 Vista general

In [ ]:
print(f"{len(df_raw):,} anuncios × {df_raw.shape[1]} columnas\n")
display(pd.crosstab(df_raw["fuente"], df_raw["tipo"], margins=True, margins_name="Total"))
display(pd.crosstab(df_raw["zona"].fillna("NA"), df_raw["tipo"], margins=True, margins_name="Total"))

_Comentario:_ TODO

### 4.2 Valores faltantes

En este dataset muchos faltantes son **estructurales** (la variable no aplica) y no aleatorios: por ejemplo, un
departamento no tiene área de terreno y una casa no tiene número de piso. Por eso se muestran por tipo de vivienda y
por fuente.

In [ ]:
cols_falt = [c for c in df_raw.columns if df_raw[c].isna().any() and c not in ("fecha_publicacion", "agencia")]

fig, axes = plt.subplots(1, 2, figsize=(14, 0.38 * len(cols_falt) + 1.5), sharey=True)
sns.heatmap(df_raw.groupby("tipo")[cols_falt].apply(lambda g: g.isna().mean() * 100).T,
            annot=True, fmt=".0f", cmap="Reds", vmin=0, vmax=100, cbar=False, ax=axes[0])
axes[0].set(title="% faltante por tipo", xlabel="", ylabel="")
fuentes_principales = ["icasas.ec", "remax.com.ec", "arriendayvende.com", "trivo.com.ec"]
sns.heatmap(df_raw[df_raw["fuente"].isin(fuentes_principales)].groupby("fuente")[cols_falt]
            .apply(lambda g: g.isna().mean() * 100).T,
            annot=True, fmt=".0f", cmap="Reds", vmin=0, vmax=100, cbar=False, ax=axes[1])
axes[1].set(title="% faltante por fuente", xlabel="", ylabel="")
plt.tight_layout(); plt.show()

_Comentario:_ TODO – p. ej. `area_terreno_m2` falta en el 100 % de departamentos (no aplica) y en ~49 % de casas;
Trivo no tiene coordenadas; `estado`/`antiguedad` dependen de la fuente. Los faltantes **no** se imputan aquí: la
imputación (con indicador de faltante) va dentro del pipeline.

### 4.3 Valores anómalos y casas renteras

Se revisan las colas de `habitaciones` y `banos`.

In [ ]:
for c in ["habitaciones", "banos", "parqueaderos"]:
    print(f"{c:13s} cola:", df_raw[c].value_counts().sort_index().tail(10).to_dict())

cols_ver = ["fuente", "tipo", "habitaciones", "banos", "area_construida_m2", "precio", "titulo"]
df_raw[(df_raw["habitaciones"] > 10) | (df_raw["banos"] > 10)][cols_ver] \
    .sort_values("habitaciones", ascending=False).head(15)

La mayoría de viviendas con más de 10 habitaciones **no son errores**: son *casas renteras* (edificios con varios
departamentos o suites para arrendar), un segmento con precio por m² distinto. Se marcan con una variable
`es_rentera` (casas cuyo título/descripción dice “rentera”, “se vende edificio” o “N departamentos independientes”,
o que tienen 10 o más habitaciones). Un primer intento con palabras sueltas como “edificio” o “rentabilidad” marcaba
también cientos de departamentos (“edificio con ascensor”), por eso el patrón es más específico. Los valores claramente imposibles se convierten a NaN (no se
elimina la fila):

| Regla | Motivo |
|---|---|
| `banos > 30` → NaN | 56 y 400 baños son errores de captura |
| departamento con `habitaciones > 10` → NaN | departamentos de 56–60 m² con 21 habitaciones |
| casa con `habitaciones = 0` → NaN | dato faltante codificado como 0 |

In [ ]:
PATRON_RENTERA = (r"\brenteras?\b|(?:vendo|venta de|se vende) edificio"
                  r"|(?:\d+|varios|dos|tres|cuatro|cinco|seis) (?:departamentos|suites|minidepartamentos) independientes")


def corregir_anomalias(df):
    df = df.copy()
    texto = (df["titulo"].fillna("") + " " + df["descripcion"].fillna("")).str.lower()
    # Casa rentera: el texto lo dice explícitamente, o tiene 10 o más habitaciones
    df["es_rentera"] = ((df["tipo"] == "casa") &
                        (texto.str.contains(PATRON_RENTERA, regex=True) | (df["habitaciones"] >= 10))).astype(int)

    reglas = {
        "banos > 30": (df["banos"] > 30, "banos"),
        "departamento con habitaciones > 10": ((df["tipo"] == "departamento") & (df["habitaciones"] > 10), "habitaciones"),
        "casa con habitaciones = 0": ((df["tipo"] == "casa") & (df["habitaciones"] == 0), "habitaciones"),
    }
    for nombre, (mask, col) in reglas.items():
        print(f"{nombre}: {mask.sum()} valores → NaN")
        df.loc[mask, col] = np.nan
    return df


df = corregir_anomalias(df_raw)
print(f"\nCasas renteras detectadas: {df['es_rentera'].sum()} "
      f"(mediana de habitaciones: {df.loc[df['es_rentera'] == 1, 'habitaciones'].median():.0f})")

### 4.4 Duplicados residuales

El dataset ya se deduplicó por tipo + precio + área + habitaciones + baños. Se buscan además anuncios con el mismo
tipo, sector, precio y área (que pueden diferir en habitaciones/baños por errores de captura entre portales).

In [ ]:
dup = df.duplicated(subset=["tipo", "sector", "precio"], keep=False) & \
      df.assign(_a=df["area_construida_m2"].round()).duplicated(subset=["tipo", "sector", "precio", "_a"], keep=False)
print(f"Anuncios en grupos de posibles duplicados: {dup.sum()}")
display(df[dup].sort_values(["precio", "sector"])[["fuente", "tipo", "sector", "precio", "area_construida_m2",
                                                   "habitaciones", "banos"]].head(10))

antes = len(df)
df = df.assign(_a=df["area_construida_m2"].round()) \
       .drop_duplicates(subset=["tipo", "sector", "precio", "_a"]).drop(columns="_a").reset_index(drop=True)
print(f"Se eliminan {antes - len(df)} duplicados → {len(df):,} filas")

### 4.5 Variable objetivo: precio

Hipótesis: el precio de una vivienda sigue aproximadamente una distribución **log-normal**. El precio se forma de
manera **multiplicativa** (área × precio por m² × factor de ubicación × factor de estado…), y el logaritmo convierte
productos en sumas; por el teorema central del límite, una suma de muchos efectos da una distribución aproximadamente
normal en escala log. Por eso los modelos hedónicos de precios de vivienda usan tradicionalmente log(precio) [12].

In [ ]:
df["log_precio"] = np.log(df[TARGET])
df["log_precio_m2"] = np.log(df["precio_m2"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
sns.histplot(data=df, x=TARGET, hue="tipo", bins=80, element="step", ax=axes[0])
axes[0].set(title="Precio (USD)", xlim=(0, df[TARGET].quantile(0.99)))
axes[0].ticklabel_format(style="plain", axis="x")
sns.histplot(data=df, x="log_precio", hue="tipo", bins=60, element="step", ax=axes[1])
axes[1].set(title="log(precio)")
stats.probplot(df["log_precio"], dist="norm", plot=axes[2])
axes[2].set_title("Q-Q plot de log(precio) vs. normal")
plt.tight_layout(); plt.show()

resumen = pd.DataFrame({
    "asimetría": [df[TARGET].skew(), df["log_precio"].skew()],
    "curtosis (exceso)": [df[TARGET].kurt(), df["log_precio"].kurt()],
}, index=["precio", "log(precio)"]).round(2)
display(resumen)
print("Percentiles de precio (USD):",
      df[TARGET].quantile([0.01, 0.25, 0.5, 0.75, 0.99]).round(-3).astype(int).to_dict())

_Comentario:_ TODO – la asimetría pasa de ~X a ~Y; el Q-Q plot es casi lineal salvo en las colas. (Con n ≈ 7000 una
prueba formal como Shapiro-Wilk rechaza la normalidad por desviaciones mínimas, por eso se juzga visualmente.)

**Decisión:** el modelo se entrena sobre **log(precio)** y se evalúa con **RMSLE** (RMSE en escala log ≈ error
relativo), coherente con lo que el modelo minimiza.

### 4.6 Precio vs. área construida

Si el precio es multiplicativo, la relación en escala log-log debe ser aproximadamente lineal. La pendiente es la
**elasticidad**: cuánto sube el precio (%) cuando el área sube 1 %.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
sns.scatterplot(data=df, x="area_construida_m2", y=TARGET, hue="tipo", alpha=0.25, s=10, ax=ax)
for i, (t, g) in enumerate(df.groupby("tipo")):
    b, a = np.polyfit(np.log(g["area_construida_m2"]), g["log_precio"], 1)
    xs = np.linspace(g["area_construida_m2"].quantile(0.01), g["area_construida_m2"].quantile(0.99), 50)
    ax.plot(xs, np.exp(a) * xs ** b, c=f"C{i}", lw=2.5, label=f"{t}: elasticidad = {b:.2f}")
    r = np.corrcoef(np.log(g["area_construida_m2"]), g["log_precio"])[0, 1]
    print(f"{t:13s} elasticidad = {b:.2f} | R² log-log = {r ** 2:.2f}")
ax.set(xscale="log", yscale="log", xlabel="Área construida (m², log)", ylabel="Precio (USD, log)",
       title="Precio vs. área (escala log-log)")
ax.legend(fontsize=8); plt.show()

_Comentario:_ TODO – una elasticidad < 1 indica que el precio por m² baja a medida que la vivienda es más grande.

### 4.7 Ubicación: zonas

In [ ]:
orden_zonas = ["quito_urbano", "valle_cumbaya_tumbaco", "valle_los_chillos", "norte_pomasqui_calderon"]
d = df[df["zona"].isin(orden_zonas)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=d, x="zona", y=TARGET, hue="tipo", order=orden_zonas, showfliers=False, ax=axes[0])
axes[0].set(title="Precio por zona", xlabel="", ylabel="USD")
sns.boxplot(data=d, x="zona", y="precio_m2", hue="tipo", order=orden_zonas, showfliers=False, ax=axes[1])
axes[1].set(title="Precio por m² por zona", xlabel="", ylabel="USD/m²")
for ax in axes:
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

display(d.pivot_table(index="zona", columns="tipo", values="precio_m2", aggfunc=["median", "count"])
        .reindex(orden_zonas).round(0))

_Comentario:_ TODO

### 4.8 Ubicación: sectores y coordenadas

`sector` tiene muchas categorías. Para decidir cómo codificarlo se analiza:
1. cuántos anuncios tiene cada sector (categorías raras → one-hot disperso);
2. cuánta variación del precio por m² explica el sector frente a la zona (η², proporción de varianza explicada por
   las medias de grupo);
3. si el precio cambia de forma **suave en el espacio** (sectores vecinos con precios parecidos). Si es así, el sector
   se puede representar por las **coordenadas de su centroide** (2 columnas numéricas), lo que además permite dar
   ubicación a los anuncios sin coordenadas.

In [ ]:
def eta2(y, grupos):
    # Proporción de la varianza de y explicada por las medias de cada grupo
    medias = y.groupby(grupos).transform("mean")
    return 1 - ((y - medias) ** 2).sum() / ((y - y.mean()) ** 2).sum()


n_sector = df["sector"].value_counts()
print(f"Sectores: {len(n_sector)} | con ≥ 10 anuncios: {(n_sector >= 10).sum()} "
      f"(cubren {n_sector[n_sector >= 10].sum() / len(df):.1%} de las filas) | con < 5: {(n_sector < 5).sum()}")

y_pm2 = df["log_precio_m2"]
print("\nVarianza de log(precio/m²) explicada (η²):")
for nombre, g in [("tipo", df["tipo"]), ("zona", df["zona"]), ("sector", df["sector"]),
                  ("tipo × zona", df["tipo"] + df["zona"].fillna("")),
                  ("tipo × sector", df["tipo"] + df["sector"].fillna(""))]:
    print(f"  {nombre:15s} η² = {eta2(y_pm2, g):.3f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
n_sector.plot.hist(bins=50, ax=ax, log=True)
ax.set(xlabel="Anuncios por sector", ylabel="Número de sectores (log)", title="Tamaño de los sectores")
plt.show()

In [ ]:
from scipy.spatial import cKDTree

# Centroide de cada sector (mediana de coordenadas de sus anuncios) y su precio/m² mediano
con_coord = df.dropna(subset=["latitud", "longitud"])
sectores = df.groupby("sector").agg(n=("precio", "size"), log_pm2=("log_precio_m2", "median")).join(
    con_coord.groupby("sector")[["latitud", "longitud"]].median())
sin_centroide = sectores["latitud"].isna()
filas_sin_coord = df["latitud"].isna()
print(f"Anuncios sin coordenadas: {filas_sin_coord.sum()} | de ellos, su sector tiene centroide: "
      f"{(~df.loc[filas_sin_coord, 'sector'].map(sectores['latitud']).isna()).sum()}")
print(f"Sectores sin ningún anuncio con coordenadas: {sin_centroide.sum()} "
      f"({sectores.loc[sin_centroide, 'n'].sum()} filas): {list(sectores.index[sin_centroide])[:10]}")

# Suavidad espacial: precio del sector vs. promedio de sus 3 sectores más cercanos
s = sectores[(sectores["n"] >= 10) & ~sin_centroide].copy()
xy = np.c_[s["latitud"], s["longitud"]]
_, vecinos = cKDTree(xy).query(xy, k=4)
s["log_pm2_vecinos"] = s["log_pm2"].values[vecinos[:, 1:]].mean(axis=1)
r = s[["log_pm2", "log_pm2_vecinos"]].corr().iloc[0, 1]
print(f"\nCorrelación precio/m² del sector vs. sus 3 vecinos más cercanos: r = {r:.2f} ({len(s)} sectores)")

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
sc = axes[0].scatter(s["longitud"], s["latitud"], c=np.exp(s["log_pm2"]), s=np.sqrt(s["n"]) * 6,
                     cmap="viridis", edgecolor="k", lw=0.3)
plt.colorbar(sc, ax=axes[0], label="Precio mediano por m² (USD)")
for nombre, fila in s.nlargest(12, "n").iterrows():
    axes[0].annotate(nombre, (fila["longitud"], fila["latitud"]), fontsize=7, xytext=(3, 3),
                     textcoords="offset points")
axes[0].set(title="Centroides de sector (tamaño ∝ nº anuncios)", xlabel="Longitud", ylabel="Latitud")
axes[0].set_aspect("equal")
axes[1].scatter(s["log_pm2_vecinos"], s["log_pm2"], alpha=0.7)
lim = [s[["log_pm2", "log_pm2_vecinos"]].min().min(), s[["log_pm2", "log_pm2_vecinos"]].max().max()]
axes[1].plot(lim, lim, "k--", lw=1)
axes[1].set(xlabel="log(precio/m²) medio de los 3 sectores vecinos", ylabel="log(precio/m²) del sector",
            title=f"Suavidad espacial del precio (r = {r:.2f})")
plt.tight_layout(); plt.show()

In [ ]:
# Coordenadas compartidas: muchos anuncios en el mismo punto = centroide geocodificado del barrio, no la casa
conteo = con_coord.groupby(["latitud", "longitud"]).size()
compartidas = conteo[conteo >= 5]
print(f"Puntos compartidos por ≥ 5 anuncios: {len(compartidas)}, que agrupan {compartidas.sum()} anuncios "
      f"({compartidas.sum() / len(con_coord):.1%} de los que tienen coordenadas)")

fig, ax = plt.subplots(figsize=(7.5, 8))
sc = ax.scatter(con_coord["longitud"], con_coord["latitud"], c=con_coord["log_precio_m2"], cmap="viridis",
                s=5, alpha=0.6)
pts = compartidas.reset_index()
ax.scatter(pts["longitud"], pts["latitud"], s=pts[0] * 2, facecolor="none", edgecolor="red", lw=0.8,
           label="punto compartido (≥ 5 anuncios)")
plt.colorbar(sc, ax=ax, label="log(precio por m²)")
ax.set(xlabel="Longitud", ylabel="Latitud", title="Anuncios con coordenadas")
ax.set_aspect("equal"); ax.legend(loc="lower left", fontsize=8)
plt.show()

_Comentario / decisión sobre `sector`:_ TODO – comparar las opciones con la evidencia de arriba:

| Opción | Columnas | Ventajas | Desventajas |
|---|---|---|---|
| One-hot (sectores raros agrupados) | ~75 | simple, interpretable | disperso; ignora que sectores vecinos se parecen |
| Coordenadas del centroide del sector | 2 | usa la suavidad espacial; ubica a los anuncios sin coordenadas; sin información del objetivo | pierde diferencias entre sectores vecinos con precios distintos |
| Target encoding | 1 | compacto, muy informativo | usa el objetivo (requiere *cross-fitting*), más difícil de explicar |

### 4.9 Fuente de los datos

`fuente` no se usa como variable predictora (es trazabilidad), pero sirve para detectar si alguna fuente es poco
confiable. Para comparar fuentes de forma justa se usa el **residuo** de log(precio/m²) respecto a la mediana de su
mismo **sector y tipo**: así no se confunde “la fuente es cara” con “la fuente tiene más anuncios en Cumbayá”.

In [ ]:
df["resid_pm2"] = df["log_precio_m2"] - df.groupby(["sector", "tipo"])["log_precio_m2"].transform("median")

d = df[df["fuente"].isin(fuentes_principales)]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.boxplot(data=d, x="fuente", y="precio_m2", hue="tipo", showfliers=False, ax=axes[0])
axes[0].set(title="Precio por m² por fuente (crudo)", xlabel="", ylabel="USD/m²")
sns.boxplot(data=d, x="fuente", y="resid_pm2", hue="tipo", showfliers=False, ax=axes[1])
axes[1].axhline(0, c="k", ls="--", lw=1)
axes[1].set(title="Residuo de log(precio/m²) dentro de sector y tipo", xlabel="", ylabel="residuo (log)")
plt.tight_layout(); plt.show()

tabla = d.groupby("fuente")["resid_pm2"].agg(["median", "count"])
tabla["diferencia %"] = (np.exp(tabla["median"]) - 1) * 100
display(tabla.round(3))

g = df.groupby("es_desarrollo")["resid_pm2"].median()
print(f"Proyectos nuevos (es_desarrollo=1) vs. resto, mismo sector y tipo: {(np.exp(g[1] - g[0]) - 1) * 100:+.0f} %")
if "fecha_publicacion" in df:
    anios = pd.to_datetime(df.loc[df["fuente"] == "arriendayvende.com", "fecha_publicacion"]).dt.year
    print("Año de publicación en arriendayvende:", anios.value_counts().sort_index().to_dict())

_Comentario / decisión:_ TODO – ¿alguna fuente se desvía tanto que convenga filtrarla? (p. ej. arriendayvende tiene
anuncios de 2021–2022). Diferencias moderadas entre portales son esperables (cada portal tiene su público); la
diferencia de los proyectos nuevos es real y se captura con `es_desarrollo`.

### 4.10 Variables numéricas

In [ ]:
num_eda = ["area_construida_m2", "area_terreno_m2", "habitaciones", "banos", "parqueaderos",
           "antiguedad_anios", "piso", "alicuota"]
fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for ax, c in zip(axes.flat, num_eda):
    x = df[c].dropna()
    usar_log = x.min() > 0 and x.skew() > 2
    sns.histplot(np.log10(x) if usar_log else x, bins=40, ax=ax)
    ax.set(title=f"{c}{' (log10)' if usar_log else ''}  — n={len(x):,}", xlabel="")
plt.tight_layout(); plt.show()

corr = df[num_eda + ["log_precio", "log_precio_m2"]].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 8})
ax.set_title("Correlación de Spearman (pares con datos disponibles)")
plt.show()

_Comentario:_ TODO – nótese la alta correlación entre área, habitaciones y baños (redundancia, relevante para la
selección de características).

### 4.11 ¿Los faltantes son informativos?

Si el precio difiere según si una variable falta o no, la ausencia contiene información y conviene agregar un
**indicador de faltante** en la imputación.

In [ ]:
filas = []
for c in ["area_terreno_m2", "parqueaderos", "antiguedad_anios", "piso", "alicuota", "estado", "banos",
          "latitud"]:
    for t, g in df.groupby("tipo"):
        falta = g[c].isna()
        if 0.02 < falta.mean() < 0.98:
            filas.append({"variable": c, "tipo": t, "% faltante": 100 * falta.mean(),
                          "Δ residuo (faltante − presente) %":
                              (np.exp(g.loc[falta, "resid_pm2"].mean() - g.loc[~falta, "resid_pm2"].mean()) - 1) * 100})
pd.DataFrame(filas).set_index(["variable", "tipo"]).round(1)

_Comentario:_ TODO

### 4.12 Amenidades y otras variables binarias

Diferencia de precio por m² entre anuncios con y sin cada característica, **dentro del mismo sector y tipo**
(media del residuo). Es una asociación, no un efecto causal: las amenidades suelen ir juntas.

In [ ]:
BINARIAS = ["piscina", "jardin", "gimnasio", "ascensor", "seguridad", "bbq", "terraza", "balcon", "bodega",
            "zona_ninos", "estacionamiento", "cocina_equipada", "calefaccion", "vista_panoramica", "patio",
            "jacuzzi_sauna", "acceso_discapacitados", "amoblado", "es_desarrollo", "precio_es_desde", "es_rentera"]
filas = []
for c in BINARIAS:
    m = df[c] == 1
    filas.append({"variable": c, "prevalencia %": 100 * m.mean(),
                  "diferencia %": (np.exp(df.loc[m, "resid_pm2"].mean() - df.loc[df[c] == 0, "resid_pm2"].mean())
                                   - 1) * 100})
amen = pd.DataFrame(filas).set_index("variable").sort_values("diferencia %")

fig, ax = plt.subplots(figsize=(8, 7))
colores = ["C3" if v < 0 else "C0" for v in amen["diferencia %"]]
ax.barh(amen.index, amen["diferencia %"], color=colores)
for i, (v, p) in enumerate(zip(amen["diferencia %"], amen["prevalencia %"])):
    ax.annotate(f"{p:.0f}%", (v, i), xytext=(3 if v >= 0 else -3, 0), textcoords="offset points",
                ha="left" if v >= 0 else "right", va="center", fontsize=7, color="gray")
ax.axvline(0, c="k", lw=1)
ax.set(xlabel="Diferencia en precio/m² (mismo sector y tipo), %",
       title="Asociación de cada variable binaria con el precio/m²\n(en gris: % de anuncios que la tienen)")
plt.show()

_Comentario:_ TODO

### 4.13 Atípicos de precio por m²

El dataset marca (sin eliminar) los anuncios con precio/m² fuera de 1,5·IQR en escala log, por tipo.

In [ ]:
atip = df[df["outlier_precio_m2"] == 1]
print(f"Atípicos marcados: {len(atip)} ({len(atip) / len(df):.1%})")
display(atip["fuente"].value_counts().to_frame("n"))
atip.sort_values("precio_m2")[["fuente", "tipo", "sector", "precio", "area_construida_m2", "precio_m2",
                                "titulo"]].iloc[np.r_[0:6, -6:0]]

_Comentario / decisión:_ TODO – revisar los extremos (¿errores de área? ¿terrenos vendidos como casas?) y decidir si
se excluyen. El modelo trabaja en escala log, lo que ya reduce su influencia.

### 4.14 Resumen del EDA y decisiones

| # | Hallazgo | Decisión para el pipeline |
|---|---|---|
| 1 | Precio muy asimétrico; log(precio) ≈ normal | Entrenar sobre log(precio); métrica RMSLE |
| 2 | Relación precio–área lineal en log-log, elasticidad < 1 | log de áreas y alícuota en el preprocesamiento común |
| 3 | Faltantes estructurales y, en algunos casos, informativos | Imputar dentro del pipeline con indicador de faltante |
| 4 | Casas renteras con muchas habitaciones | Variable `es_rentera`; anomalías → NaN |
| 5 | Sector explica gran parte de la varianza; precio espacialmente suave (r ≈ 0,77) | One-hot de `sector` + coordenadas (faltantes → centroide del sector) + distancia a La Carolina |
| 6 | Proyectos nuevos ~X % más caros en el mismo sector | Incluir `es_desarrollo` |
| 7 | Diferencias moderadas entre fuentes en el mismo sector y tipo | No se filtra ninguna fuente; `fuente` no es predictor |
| 8 | TODO | TODO |

## 5. Preparación para el modelado

Decisiones tomadas en el EDA (sección 4.14):

- **Objetivo en escala log.** `y = log(precio)` se calcula **antes** de la partición. Es una transformación fija (no
  aprende nada de los datos), así que no hay *data leakage*. La métrica de validación es el RMSE sobre log(precio),
  es decir el **RMSLE**, que equivale aproximadamente a un error relativo. Solo al final (sección 11) se vuelve a USD.
- **Alcance:** se excluyen las 7 filas fuera de Quito y sus valles (`zona` = `rural_otro` o vacía).
- **Variables** (las mismas para todos los modelos):

| Grupo | Columnas | Transformación (dentro del pipeline) |
|---|---|---|
| Tamaño (asimétricas) | `area_construida_m2`, `area_terreno_m2`, `alicuota` | log → imputación (mediana) + indicador de faltante → escalado |
| Numéricas | `habitaciones`, `banos`, `parqueaderos`, `antiguedad_anios`, `piso` | imputación (mediana) + indicador de faltante → escalado |
| Ubicación | `latitud`, `longitud` (+ `sector`) | coordenadas faltantes → centroide del sector; + distancia a La Carolina → escalado |
| Binarias | amenidades, `es_desarrollo`, `precio_es_desde`, `es_rentera` | imputación (moda) |
| Categóricas | `sector`, `tipo`, `zona`, `estado`, `canton` | faltante → "desconocido" → one-hot (categorías con < 10 anuncios agrupadas) |

**No se usan como predictores:** `precio_m2`, `outlier_precio_m2` y las columnas auxiliares del EDA (se calculan con
el precio: *data leakage*), `fuente` (solo trazabilidad), `id`, `url`, textos y fechas.

**Feature extraction:** no aplica: los datos son tabulares (estructurados) y no se usa *transfer learning*.

In [ ]:
df_modelo = df[df["zona"].isin(["quito_urbano", "valle_cumbaya_tumbaco", "valle_los_chillos",
                                "norte_pomasqui_calderon"])].reset_index(drop=True)
print(f"Filas para el modelado: {len(df_modelo):,} (se excluyen {len(df) - len(df_modelo)} fuera del alcance)")

LOG_COLS = ["area_construida_m2", "area_terreno_m2", "alicuota"]
NUM_COLS = ["habitaciones", "banos", "parqueaderos", "antiguedad_anios", "piso"]
UBIC_COLS = ["latitud", "longitud", "sector"]
BIN_COLS = BINARIAS                     # definidas en la sección 4.12
CAT_COLS = ["sector", "tipo", "zona", "estado", "canton"]
FEATURES = list(dict.fromkeys(LOG_COLS + NUM_COLS + UBIC_COLS + BIN_COLS + CAT_COLS))

X = df_modelo[FEATURES]
y = np.log(df_modelo[TARGET])           # objetivo en escala log (transformación fija, sin leakage)
print(f"{X.shape[1]} columnas de entrada")

## 6. Partición entrenamiento / prueba

Se reserva un 20 % de los datos como **conjunto de prueba**, que no se toca hasta la evaluación final. La partición se
estratifica por deciles de precio para que ambos conjuntos tengan la misma distribución del objetivo. Toda la
optimización y la comparación estadística se hacen con validación cruzada **sobre el conjunto de entrenamiento**.

_Limitación conocida:_ las unidades de un mismo proyecto de constructora (p. ej. 75 tipologías del proyecto Beat)
pueden quedar en folds distintos, lo que hace la validación algo optimista para esas filas. Se acepta para mantener el
esquema estándar de *repeated k-fold*.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=pd.qcut(y, 10, labels=False))
print(f"Entrenamiento: {len(X_train):,} | Prueba: {len(X_test):,}")

## 7. Pipeline

Todo el flujo se encapsula en un `Pipeline` de `scikit-learn` [1]. La imputación, el escalado, la codificación, los
centroides de sector y la selección de características se **ajustan solo con los datos de entrenamiento de cada
fold**, sin *data leakage*. Los tres modelos reciben exactamente la misma entrada:

```
Pipeline
├── preprocesamiento (ColumnTransformer)
│   ├── tamaño:      log1p → SimpleImputer(mediana, indicador) → StandardScaler
│   ├── numéricas:   SimpleImputer(mediana, indicador) → StandardScaler
│   ├── ubicación:   ImputadorCoordenadas (centroide del sector + distancia a La Carolina) → StandardScaler
│   ├── binarias:    SimpleImputer(moda)
│   └── categóricas: SimpleImputer("desconocido") → OneHotEncoder(min_frequency=10)
├── varianza:  VarianceThreshold          ← descarta columnas constantes en el fold
├── seleccion: SelectPercentile(f_regression)   ← feature selection tipo filtro
└── modelo:    MLPRegressor (con objetivo estandarizado) | RandomForestRegressor | Ridge (referencia)
```

**Feature selection (filtro):** `SelectPercentile` con `f_regression` calcula, para cada característica, el
estadístico F de una regresión lineal univariada contra log(precio) y conserva el percentil superior. Es un método de
tipo filtro porque evalúa cada variable por separado, sin usar el modelo. El percentil se optimiza con Optuna como un
hiperparámetro más.

**Logaritmos en variables:** el log de las áreas y la alícuota es necesario para Ridge (modelo hedónico log-log:
el coeficiente del área se interpreta como elasticidad) y ayuda a la MLP. No afecta al Random Forest, porque una
transformación monótona no cambia dónde corta un árbol. Así los tres modelos compiten con las mismas entradas.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

CAROLINA = (-0.1807, -78.4848)   # Parque La Carolina, centro financiero de Quito


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))


class ImputadorCoordenadas(BaseEstimator, TransformerMixin):
    # Completa latitud/longitud faltantes con el centroide (mediana) del sector, aprendido SOLO en entrenamiento.
    # Si el sector no tiene centroide, usa la mediana global. Agrega la distancia al Parque La Carolina.
    # Entrada: columnas [latitud, longitud, sector]. Salida: [latitud, longitud, dist_carolina_km].
    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=UBIC_COLS)
        con = X.dropna(subset=["latitud", "longitud"])
        self.centroides_ = con.groupby("sector")[["latitud", "longitud"]].median()
        self.global_ = con[["latitud", "longitud"]].median()
        return self

    def transform(self, X):
        X = pd.DataFrame(X, columns=UBIC_COLS).reset_index(drop=True)
        c = X[["sector"]].join(self.centroides_, on="sector")
        lat = X["latitud"].fillna(c["latitud"]).fillna(self.global_["latitud"])
        lon = X["longitud"].fillna(c["longitud"]).fillna(self.global_["longitud"])
        return np.c_[lat, lon, haversine_km(lat, lon, *CAROLINA)].astype(float)

    def get_feature_names_out(self, input_features=None):
        return np.array(["latitud", "longitud", "dist_carolina_km"])


def crear_preprocesador():
    tamano = Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                       ("imputar", SimpleImputer(strategy="median", add_indicator=True)),
                       ("escalar", StandardScaler())])
    num = Pipeline([("imputar", SimpleImputer(strategy="median", add_indicator=True)),
                    ("escalar", StandardScaler())])
    ubic = Pipeline([("coordenadas", ImputadorCoordenadas()), ("escalar", StandardScaler())])
    binarias = SimpleImputer(strategy="most_frequent")
    cat = Pipeline([("imputar", SimpleImputer(strategy="constant", fill_value="desconocido")),
                    ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=10,
                                             sparse_output=False))])
    return ColumnTransformer([("tam", tamano, LOG_COLS), ("num", num, NUM_COLS), ("ubic", ubic, UBIC_COLS),
                              ("bin", binarias, BIN_COLS), ("cat", cat, CAT_COLS)],
                             verbose_feature_names_out=False)


MODELOS_BASE = {
    # La MLP aprende mejor con el objetivo estandarizado (media 0, desviación 1): log(precio) ≈ 12 obligaría a la red
    # a gastar épocas aprendiendo el desplazamiento. El StandardScaler del objetivo se ajusta dentro de cada fold.
    "MLP": TransformedTargetRegressor(
        regressor=MLPRegressor(hidden_layer_sizes=(64, 32), early_stopping=True, n_iter_no_change=20, max_iter=1000,
                               random_state=SEED),
        transformer=StandardScaler()),
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=1),
    "Ridge": Ridge(),
}


def crear_pipeline(nombre_modelo):
    return Pipeline([
        ("preprocesamiento", crear_preprocesador()),
        ("varianza", VarianceThreshold()),
        ("seleccion", SelectPercentile(f_regression, percentile=100)),
        ("modelo", clone(MODELOS_BASE[nombre_modelo])),
    ])


crear_pipeline("MLP")

In [ ]:
# Línea base: modelos con hiperparámetros por defecto (RMSLE = RMSE sobre log(precio))
cv_rapido = RepeatedKFold(n_splits=5, n_repeats=1, random_state=SEED)
for nombre in MODELOS_BASE:
    s = -cross_val_score(crear_pipeline(nombre), X_train, y_train, cv=cv_rapido, scoring=SCORING, n_jobs=N_JOBS)
    print(f"{nombre:13s} RMSLE = {s.mean():.4f} ± {s.std():.4f}  (≈ {100 * (np.exp(s.mean()) - 1):.0f} % de error típico)")

In [ ]:
# Puntajes del filtro f_regression sobre todo el conjunto de entrenamiento (solo para interpretación)
pre = Pipeline([("preprocesamiento", crear_preprocesador()), ("varianza", VarianceThreshold())]).fit(X_train)
nombres_pre = pre[0].get_feature_names_out()[pre[1].get_support()]
F, _ = f_regression(pre.transform(X_train), y_train)
puntajes = pd.Series(F, index=nombres_pre).sort_values(ascending=False)
print(f"{len(puntajes)} características después del preprocesamiento")

fig, ax = plt.subplots(figsize=(8, 0.25 * 30 + 1))
puntajes.head(30).plot.barh(ax=ax)
ax.invert_yaxis()
ax.set(xlabel="Estadístico F (f_regression)", title="Top 30 características según el filtro")
plt.show()

## 8. Optimización de hiperparámetros con Optuna

Se optimizan los dos modelos de la comparación con Optuna [2] (muestreador TPE). Cada *trial* se evalúa con
**repeated k-fold CV** (`CV_OPT`) sobre el conjunto de entrenamiento, y se minimiza el **RMSLE**:

- **MLP** (perceptrón multicapa [7], optimizador Adam [9]): número de capas ocultas (1–2), neuronas por capa,
  regularización L2 (`alpha`), tasa de aprendizaje y tamaño de lote. Usa *early stopping* con el 10 % del fold de
  entrenamiento, y una red pequeña, porque hay ~5.600 filas de entrenamiento. El objetivo se estandariza dentro del
  modelo: en una prueba preliminar (5 folds) esto bajó el RMSLE de la MLP por defecto de 0,340 a 0,301.
- **Random Forest** [8]: número de árboles, profundidad máxima, mínimo de muestras por hoja y fracción de
  variables por división.
- **Ridge** [13] (referencia): solo `alpha`. Es el modelo hedónico lineal en log-log [12].

El percentil de la selección de características se optimiza junto con los hiperparámetros de cada modelo.

In [ ]:
# Cada espacio devuelve directamente los parámetros del pipeline ("paso__parámetro")
def espacio_mlp(t):
    capas = [t.suggest_int("neuronas_1", 16, 256, log=True)]
    if t.suggest_int("n_capas", 1, 2) == 2:
        capas.append(t.suggest_int("neuronas_2", 8, 128, log=True))
    return {"seleccion__percentile": t.suggest_int("percentile", 30, 100, step=10),
            "modelo__regressor__hidden_layer_sizes": tuple(capas),
            "modelo__regressor__alpha": t.suggest_float("alpha", 1e-5, 10, log=True),
            "modelo__regressor__learning_rate_init": t.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
            "modelo__regressor__batch_size": t.suggest_categorical("batch_size", [32, 64, 128])}


def espacio_rf(t):
    return {"seleccion__percentile": t.suggest_int("percentile", 30, 100, step=10),
            "modelo__n_estimators": t.suggest_int("n_estimators", 100, 500, step=50),
            "modelo__max_depth": t.suggest_int("max_depth", 4, 32, log=True),
            "modelo__min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 20, log=True),
            "modelo__max_features": t.suggest_float("max_features", 0.1, 1.0)}


def espacio_ridge(t):
    return {"seleccion__percentile": t.suggest_int("percentile", 30, 100, step=10),
            "modelo__alpha": t.suggest_float("alpha", 1e-3, 1e3, log=True)}


ESPACIOS = {"MLP": espacio_mlp, "RandomForest": espacio_rf, "Ridge": espacio_ridge}
N_TRIALS_MODELO = {"MLP": N_TRIALS, "RandomForest": N_TRIALS, "Ridge": max(10, N_TRIALS // 3)}


def optimizar(nombre_modelo):
    def objetivo(trial):
        modelo = crear_pipeline(nombre_modelo).set_params(**ESPACIOS[nombre_modelo](trial))
        scores = -cross_val_score(modelo, X_train, y_train, cv=CV_OPT, scoring=SCORING, n_jobs=N_JOBS)
        trial.set_user_attr("rmsle_std", scores.std())
        return scores.mean()

    estudio = optuna.create_study(direction="minimize", study_name=nombre_modelo,
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
    estudio.optimize(objetivo, n_trials=N_TRIALS_MODELO[nombre_modelo], show_progress_bar=True)
    print(f"{nombre_modelo}: mejor RMSLE CV = {estudio.best_value:.4f} | {estudio.best_params}")
    return estudio


estudios = {nombre: optimizar(nombre) for nombre in ESPACIOS}

In [ ]:
from optuna.visualization import matplotlib as ovm

PRINCIPALES = ["MLP", "RandomForest"]   # los dos modelos de la comparación estadística
for nombre in PRINCIPALES:
    ax = ovm.plot_optimization_history(estudios[nombre])
    ax.set_title(f"Historial de optimización – {nombre}")
    ax.set_ylabel("RMSLE CV")
    plt.show()

In [ ]:
for nombre in PRINCIPALES:
    ax = ovm.plot_param_importances(estudios[nombre])
    ax.set_title(f"Importancia de hiperparámetros – {nombre}")
    plt.show()

**Gráfica de la optimización de hiperparámetros:** cada punto es un *trial* de Optuna; el eje y es el RMSLE de
validación cruzada en función del valor de cada hiperparámetro (el color indica el número de trial).

In [ ]:
from matplotlib.ticker import NullFormatter

for nombre in PRINCIPALES:
    axes = ovm.plot_slice(estudios[nombre])
    for ax in np.atleast_1d(axes):
        ax.xaxis.set_minor_formatter(NullFormatter())   # evita etiquetas superpuestas en ejes log
        ax.tick_params(axis="x", labelsize=8)
    plt.suptitle(f"RMSLE CV vs. hiperparámetros – {nombre}", y=1.02)
    plt.show()

In [ ]:
import json

# Mejores pipelines: se reconstruyen los parámetros con FixedTrial (maneja la capa oculta opcional de la MLP)
mejores = {n: crear_pipeline(n).set_params(**ESPACIOS[n](optuna.trial.FixedTrial(e.best_params)))
           for n, e in estudios.items()}

json.dump({n: e.best_params for n, e in estudios.items()},
          open(RESULTADOS / "mejores_hiperparametros.json", "w"), indent=2)
for n, e in estudios.items():
    e.trials_dataframe().to_csv(RESULTADOS / f"optuna_trials_{n}.csv", index=False)

pd.DataFrame({n: {"RMSLE CV": e.best_value, "Desv. est.": e.best_trial.user_attrs["rmsle_std"], **e.best_params}
              for n, e in estudios.items()}).T

_Comentario:_ TODO – qué hiperparámetros resultaron más importantes y en qué rangos se encuentran los óptimos.

## 9. Curvas de validación y de aprendizaje

### 9.1 Curvas de validación

Se varía un hiperparámetro de cada modelo optimizado (los demás quedan en su valor óptimo) y se grafica el error de
entrenamiento y de validación (repeated k-fold CV). Permite ver el compromiso sesgo–varianza: en la MLP, la
regularización L2 (`alpha`); en el Random Forest, la profundidad máxima de los árboles.

In [ ]:
def a_rmsle(s):
    return -s.mean(axis=1), s.std(axis=1)


curvas = {
    "MLP": ("modelo__regressor__alpha", np.logspace(-5, 1, 9), "alpha (regularización L2)", "alpha", True),
    "RandomForest": ("modelo__max_depth", np.array([2, 3, 4, 6, 8, 12, 16, 24, 32]), "max_depth", "max_depth", False),
}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, (nombre, (param, rango, etiqueta, clave, escala_log)) in zip(axes, curvas.items()):
    tr, va = validation_curve(mejores[nombre], X_train, y_train, param_name=param, param_range=rango,
                              cv=CV_OPT, scoring=SCORING, n_jobs=N_JOBS)
    (tr_m, tr_s), (va_m, va_s) = a_rmsle(tr), a_rmsle(va)
    ax.plot(rango, tr_m, "o-", label="Entrenamiento")
    ax.fill_between(rango, tr_m - tr_s, tr_m + tr_s, alpha=0.2)
    ax.plot(rango, va_m, "o-", label="Validación (CV)")
    ax.fill_between(rango, va_m - va_s, va_m + va_s, alpha=0.2)
    ax.axvline(estudios[nombre].best_params[clave], ls="--", c="gray", label="Óptimo Optuna")
    ax.set(xscale="log" if escala_log else "linear", xlabel=etiqueta, ylabel="RMSLE",
           title=f"Curva de validación – {nombre}")
    ax.legend()
plt.tight_layout(); plt.show()

_Comentario:_ TODO

### 9.2 Curvas de aprendizaje

Error de entrenamiento y validación en función del número de muestras de entrenamiento. Responde si el tamaño del
dataset limita a cada modelo: si la curva de validación sigue bajando al usar el 100 % de los datos, más datos
ayudarían (pregunta especialmente relevante para la MLP).

In [ ]:
fig, axes = plt.subplots(1, len(PRINCIPALES), figsize=(14, 4.8), sharey=True)
for ax, nombre in zip(axes, PRINCIPALES):
    tam, tr, va = learning_curve(mejores[nombre], X_train, y_train, train_sizes=np.linspace(0.1, 1.0, 8),
                                 cv=CV_OPT, scoring=SCORING, n_jobs=N_JOBS, shuffle=True, random_state=SEED)
    (tr_m, tr_s), (va_m, va_s) = a_rmsle(tr), a_rmsle(va)
    ax.plot(tam, tr_m, "o-", label="Entrenamiento")
    ax.fill_between(tam, tr_m - tr_s, tr_m + tr_s, alpha=0.2)
    ax.plot(tam, va_m, "o-", label="Validación (CV)")
    ax.fill_between(tam, va_m - va_s, va_m + va_s, alpha=0.2)
    ax.set(title=f"Curva de aprendizaje – {nombre}", xlabel="Muestras de entrenamiento", ylabel="RMSLE")
    ax.legend()
plt.tight_layout(); plt.show()

_Comentario:_ TODO

## 10. Comparación estadística: MLP vs. Random Forest

**Pregunta:** en este dataset tabular y relativamente pequeño, ¿una red neuronal (MLP) predice igual de bien que un
ensamble de árboles (Random Forest)? La literatura reporta que en datos tabulares de tamaño medio los ensambles de
árboles suelen superar a las redes neuronales [14]; aquí se pone a prueba con nuestros datos.

Ambos modelos optimizados se evalúan con **repeated k-fold CV** (`CV_COMP`: 10 repeticiones × 10 folds = 100
estimaciones del RMSLE por modelo) **en exactamente los mismos folds** (misma semilla): las muestras son **pareadas**.

**Hipótesis:**
- $H_0$: ambos modelos tienen el mismo desempeño (la diferencia de RMSLE por fold está centrada en 0).
- $H_1$: el desempeño es distinto.

Nivel de significancia: $\alpha = 0.05$.

**Pruebas utilizadas:**
1. **Wilcoxon de rangos con signo** [5]: no paramétrica, no asume normalidad de las diferencias; recomendada por
   Demšar [6] para comparar dos modelos.
2. **t-test pareado con corrección de Nadeau y Bengio** [4]: en *k-fold* repetido los conjuntos de entrenamiento se
   solapan, por lo que las 100 estimaciones no son independientes y el t-test clásico subestima la varianza
   (demasiados falsos positivos). La corrección infla la varianza por el factor $\left(\frac{1}{k} +
   \frac{n_{test}}{n_{train}}\right)$.

Ridge (modelo hedónico lineal) se incluye como **referencia**: muestra cuánto gana cada modelo no lineal sobre el
modelo clásico, pero no entra en la prueba de hipótesis.

In [ ]:
comparar = {"MLP": mejores["MLP"], "RandomForest": mejores["RandomForest"], "Ridge (referencia)": mejores["Ridge"]}

rmsle_cv = {}
for nombre, modelo in comparar.items():
    rmsle_cv[nombre] = -cross_val_score(modelo, X_train, y_train, cv=CV_COMP, scoring=SCORING, n_jobs=N_JOBS)
    print(f"{nombre:20s} RMSLE = {rmsle_cv[nombre].mean():.4f} ± {rmsle_cv[nombre].std():.4f} "
          f"({len(rmsle_cv[nombre])} folds)")

res_cv = pd.DataFrame(rmsle_cv)
res_cv.to_csv(RESULTADOS / "rmsle_repeated_kfold.csv", index_label="fold")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.boxplot(data=res_cv, ax=axes[0])
axes[0].set(ylabel="RMSLE", title=f"RMSLE en repeated k-fold ({len(res_cv)} folds)")

dif = res_cv["MLP"] - res_cv["RandomForest"]
sns.histplot(dif, bins=25, kde=True, ax=axes[1])
axes[1].axvline(0, c="k", ls="--")
axes[1].set(xlabel="RMSLE(MLP) − RMSLE(RF)", title="Diferencias pareadas por fold")
plt.tight_layout(); plt.show()

In [ ]:
a, b = res_cv["MLP"].values, res_cv["RandomForest"].values
d = a - b
ALPHA = 0.05

# Normalidad de las diferencias (para decidir qué prueba es más apropiada)
sh = stats.shapiro(d)
print(f"Shapiro-Wilk sobre las diferencias: W = {sh.statistic:.3f}, p = {sh.pvalue:.4f}")

# 1) Wilcoxon de rangos con signo
w = stats.wilcoxon(a, b)
print(f"Wilcoxon: estadístico = {w.statistic:.1f}, p = {w.pvalue:.4g}")


# 2) t-test pareado corregido (Nadeau & Bengio, 2003)
def t_test_corregido(a, b, n_train, n_test):
    d = a - b
    k = len(d)
    var = d.var(ddof=1)
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * var)
    p = 2 * stats.t.sf(abs(t), df=k - 1)
    return t, p


n_splits = CV_COMP.cvargs["n_splits"]
n_test_fold = len(X_train) / n_splits
n_train_fold = len(X_train) - n_test_fold
t_c, p_c = t_test_corregido(a, b, n_train_fold, n_test_fold)
t_n, p_n = stats.ttest_rel(a, b)
print(f"t-test pareado corregido: t = {t_c:.3f}, p = {p_c:.4g}")
print(f"(t-test pareado sin corregir, solo como referencia: t = {t_n:.3f}, p = {p_n:.4g})")

mejor = "MLP" if d.mean() < 0 else "RandomForest"
print(f"\nDiferencia media de RMSLE (MLP − RF): {d.mean():+.4f}; modelo con menor error: {mejor}")
for nombre, p in [("Wilcoxon", w.pvalue), ("t corregido", p_c)]:
    decision = "se RECHAZA H0" if p < ALPHA else "NO se rechaza H0"
    print(f"{nombre}: p = {p:.4g} → {decision} (α = {ALPHA})")

_Interpretación:_ TODO – indicar si la diferencia es estadísticamente significativa, cuál modelo es mejor y si la
diferencia es relevante en la práctica (una diferencia de 0,01 en RMSLE ≈ 1 punto porcentual de error).

> Observar que el Wilcoxon y el t-test sin corregir suelen dar p-valores mucho menores que el t-test corregido:
> ninguno de los dos considera la dependencia entre folds. Si las pruebas discrepan, la conclusión más
> conservadora (y defendible) es la del t-test corregido.

> **Nota metodológica:** los hiperparámetros se optimizaron con CV sobre el mismo conjunto de entrenamiento, por lo
> que los RMSLE de esta sección pueden ser ligeramente optimistas. Esto afecta por igual a ambos modelos, así que la
> comparación sigue siendo justa; la estimación no sesgada del error es la del conjunto de prueba (sección 11).

## 11. Evaluación final en el conjunto de prueba

Los modelos se reentrenan con todo el conjunto de entrenamiento y se evalúan una sola vez en el conjunto de prueba.
Además del RMSLE (escala del entrenamiento), las predicciones se vuelven a USD con $e^{\hat{y}}$ para reportar
errores interpretables.

In [ ]:
def metricas(y_log, pred_log):
    real, pred = np.exp(y_log), np.exp(pred_log)
    return {"RMSLE": root_mean_squared_error(y_log, pred_log),
            "R² (log)": r2_score(y_log, pred_log),
            "MAE (USD)": mean_absolute_error(real, pred),
            "Mediana |error| (USD)": np.median(np.abs(real - pred)),
            "MAPE (%)": 100 * mean_absolute_percentage_error(real, pred),
            "RMSE (USD)": root_mean_squared_error(real, pred)}


predicciones, tabla = {}, {}
for nombre, modelo in comparar.items():
    modelo.fit(X_train, y_train)
    predicciones[nombre] = modelo.predict(X_test)
    tabla[nombre] = metricas(y_test, predicciones[nombre])

pd.DataFrame(tabla).T.to_csv(RESULTADOS / "metricas_test.csv")
pd.DataFrame(tabla).T.style.format({"RMSLE": "{:.4f}", "R² (log)": "{:.3f}", "MAE (USD)": "{:,.0f}",
                                    "Mediana |error| (USD)": "{:,.0f}", "MAPE (%)": "{:.1f}",
                                    "RMSE (USD)": "{:,.0f}"})

In [ ]:
final = mejor   # modelo con menor error en la comparación (sección 10)
real, pred = np.exp(y_test), np.exp(predicciones[final])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(real, pred, s=10, alpha=0.4)
lim = [min(real.min(), pred.min()), max(real.max(), pred.max())]
axes[0].plot(lim, lim, "k--", lw=1)
axes[0].set(xscale="log", yscale="log", xlabel="Precio real (USD)", ylabel="Precio predicho (USD)",
            title=f"Real vs. predicho – {final}")

err_rel = (pred - real) / real * 100
sns.boxplot(x=X_test["zona"].values, y=err_rel.values, hue=X_test["tipo"].values, showfliers=False, ax=axes[1])
axes[1].axhline(0, c="k", ls="--")
axes[1].set(xlabel="", ylabel="Error relativo (%)", title="Error relativo por zona y tipo")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

In [ ]:
# Características que sobrevivieron al filtro en el modelo final
pipe_final = comparar[final]
nombres = pipe_final.named_steps["preprocesamiento"].get_feature_names_out()
nombres = nombres[pipe_final.named_steps["varianza"].get_support()]
mask = pipe_final.named_steps["seleccion"].get_support()
print(f"Características seleccionadas: {mask.sum()} de {len(mask)}")
print("Descartadas por el filtro:", list(nombres[~mask]))

_Comentario:_ TODO

## 12. Visualización del espacio de características con t-SNE

Se proyecta a 2 dimensiones con t-SNE [3] el espacio de características **después del preprocesamiento y la
selección** del modelo final (lo que realmente “ve” el modelo). Como el problema es de regresión, cada punto se colorea
por su **precio** (escala log); en el segundo panel, por zona.

In [ ]:
N_TSNE = 3000
idx = rng.choice(len(X_train), size=min(N_TSNE, len(X_train)), replace=False)
X_sub, y_sub = X_train.iloc[idx], y_train.iloc[idx]

Z = pipe_final[:-1].transform(X_sub)   # preprocesamiento + varianza + selección (sin el modelo)
emb = TSNE(n_components=2, perplexity=30, init="pca", random_state=SEED).fit_transform(Z)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc = axes[0].scatter(emb[:, 0], emb[:, 1], c=y_sub / np.log(10), cmap="viridis", s=8, alpha=0.8)
plt.colorbar(sc, ax=axes[0], label="log10(precio USD)")
axes[0].set_title("t-SNE coloreado por precio")
sns.scatterplot(x=emb[:, 0], y=emb[:, 1], hue=X_sub["zona"].values, s=8, alpha=0.8, ax=axes[1])
axes[1].set_title("t-SNE coloreado por zona")
for ax in axes:
    ax.set(xlabel="t-SNE 1", ylabel="t-SNE 2")
plt.tight_layout(); plt.show()

_Comentario:_ TODO – ¿se observan gradientes de precio o grupos (p. ej. valles vs. ciudad, casas vs.
departamentos)? Recordar que en t-SNE las distancias entre grupos lejanos no son interpretables, solo la estructura
local.

## 13. Conclusiones

_(Las conclusiones deben basarse en los resultados obtenidos, no en teoría.)_

1. TODO – desempeño del mejor modelo en el conjunto de prueba (RMSLE, MAPE, error mediano en USD) y cuánto mejora
   sobre el modelo hedónico lineal (Ridge).
2. TODO – resultado de la comparación estadística (¿diferencia significativa? ¿relevante en la práctica?).
3. TODO – variables más influyentes según la selección de características y el EDA.
4. TODO – diferencias Quito vs. valles observadas en los datos y en los errores del modelo.
5. TODO – qué indican las curvas de aprendizaje (¿le faltan datos a la MLP?).
6. TODO – limitaciones (precios de oferta vs. precios de cierre, sesgo de la fuente, datos faltantes) y trabajo futuro.

## 14. Referencias

[1] F. Pedregosa *et al.*, “Scikit-learn: Machine learning in Python,” *J. Mach. Learn. Res.*, vol. 12,
pp. 2825–2830, 2011.

[2] T. Akiba, S. Sano, T. Yanase, T. Ohta, and M. Koyama, “Optuna: A next-generation hyperparameter optimization
framework,” in *Proc. 25th ACM SIGKDD Int. Conf. Knowl. Discov. Data Min.*, 2019, pp. 2623–2631.

[3] L. van der Maaten and G. Hinton, “Visualizing data using t-SNE,” *J. Mach. Learn. Res.*, vol. 9,
pp. 2579–2605, 2008.

[4] C. Nadeau and Y. Bengio, “Inference for the generalization error,” *Mach. Learn.*, vol. 52, no. 3,
pp. 239–281, 2003.

[5] F. Wilcoxon, “Individual comparisons by ranking methods,” *Biometrics Bull.*, vol. 1, no. 6, pp. 80–83, 1945.

[6] J. Demšar, “Statistical comparisons of classifiers over multiple data sets,” *J. Mach. Learn. Res.*, vol. 7,
pp. 1–30, 2006.

[7] D. E. Rumelhart, G. E. Hinton, and R. J. Williams, “Learning representations by back-propagating errors,”
*Nature*, vol. 323, no. 6088, pp. 533–536, 1986.

[8] L. Breiman, “Random forests,” *Mach. Learn.*, vol. 45, no. 1, pp. 5–32, 2001.

[9] D. P. Kingma and J. Ba, “Adam: A method for stochastic optimization,” in *Proc. 3rd Int. Conf. Learn.
Represent. (ICLR)*, 2015.

[10] Dataset propio construido a partir de: icasas.ec, “Casas y departamentos en venta en Quito”; RE/MAX Ecuador,
“Comprar propiedades”; Arrienda y Vende, “Casas de venta en Quito”; Trivo, “Casas en venta en Quito”, 2026.
Accedido: 11–12 nov. 2026. Referencias completas en `data/README.md`.

[11] scikit-learn developers, “Plot validation and learning curves,” scikit-learn documentation. [Online].
Available: https://scikit-learn.org/stable/modules/learning_curve.html

[12] S. Rosen, “Hedonic prices and implicit markets: Product differentiation in pure competition,” *J. Polit. Econ.*,
vol. 82, no. 1, pp. 34–55, 1974.

[13] A. E. Hoerl and R. W. Kennard, “Ridge regression: Biased estimation for nonorthogonal problems,”
*Technometrics*, vol. 12, no. 1, pp. 55–67, 1970.

[14] L. Grinsztajn, E. Oyallon, and G. Varoquaux, “Why do tree-based models still outperform deep learning on
typical tabular data?,” in *Adv. Neural Inf. Process. Syst. 35*, 2022, pp. 507–520.